# Vehicle Re-ID: OSNet-AIN → VeRi-776 → CityFlowV2-ReID → Parking-140

**Цель:** обучить эмбеддинг для межкамерного поиска автомобиля и отдельно проверить open-set отказ.

Порядок эксперимента:
1. ImageNet-pretrained OSNet-AIN → supervised обучение на VeRi-776.
2. Fine-tuning на AI City Challenge 2021 Track 2 (CityFlowV2-ReID). Используется только `image_train` + `train_label.xml` + `train_track.txt`.
3. Fine-tuning на Parking-140 — целевой домен парковки.

Методические ограничения:
- Валидация AI City и Parking делится по целым vehicle ID; ID из validation не участвуют в соответствующем обучении.
- В AI City query и gallery для каждого validation ID выбираются из разных камер.
- P×K sampler сначала выбирает разные камеры, затем разные треки одного ID.
- Dataset ID имеют отдельные пространства имён (`veri_`, `aicity_`, `parking_`).
- Используется CE + batch-hard triplet, AdamW и cosine scheduler.
- Для VeRi retrieval исключаются same-ID/same-camera совпадения.
- Сохраняется лучший checkpoint по cross-camera mAP.

> Важно: это экспериментальный pipeline, а не гарантия роста качества. Сравнивай все этапы на одинаковом Parking validation split и затем проверяй false accept на реальных переходах камер.

## 0. Среда и зависимости

Запускай в Google Colab с GPU. Если работаешь локально, адаптируй пути и не выполняй `drive.mount`.

In [ ]:
!pip -q install cython
!pip -q install --no-build-isolation git+https://github.com/KaiyangZhou/deep-person-reid.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, glob, random, shutil, xml.etree.ElementTree as ET, math, copy, json, time
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
import torchreid

SEED = 42
VAL_FRACTION = 0.20  # доля identity-disjoint validation для AI City и Parking
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

# Измени только эти пути при необходимости.
VERI_ROOT = '/content/VeRi'
AICITY_ROOT = '/content/CityFlowV2-ReID'  # папка с image_train/, train_label.xml и train_track.txt
PARKING_ROOT = '/content/reid_dataset'
SAVE_ROOT = '/content/drive/MyDrive/prod/reid_training/osnet_ain_veri_aicity_parking'
os.makedirs(SAVE_ROOT, exist_ok=True)

## 1. Данные VeRi-776

Ожидается структура `VeRi/train_label.xml`, `image_train/`, `image_query/`, `image_test/`. Если архив уже лежит в другом месте, просто укажи правильный `VERI_ROOT`. Скачивание не встроено намеренно: ссылки на зеркала датасетов могут меняться, а локальный архив надёжнее.

In [ ]:
assert os.path.isdir(VERI_ROOT), f'Не найдена папка VeRi: {VERI_ROOT}'
assert os.path.isfile(os.path.join(VERI_ROOT, 'train_label.xml'))
assert os.path.isdir(os.path.join(VERI_ROOT, 'image_train'))
assert os.path.isdir(os.path.join(VERI_ROOT, 'image_query'))
assert os.path.isdir(os.path.join(VERI_ROOT, 'image_test'))

def read_veri_train(xml_path, image_dir):
    with open(xml_path, 'r', encoding='gb2312', errors='ignore') as f:
        root = ET.fromstring(f.read())
    out=[]; missing=0
    items = root.find('Items')
    if items is None: raise RuntimeError('В train_label.xml не найден узел Items')
    for item in items:
        name = item.attrib.get('imageName')
        pid = item.attrib.get('vehicleID')
        cam = item.attrib.get('cameraID', '')
        if not name or pid is None: continue
        m = re.search(r'(?:c)?(\d+)', cam)
        camid = int(m.group(1)) if m else 0
        path = os.path.join(image_dir, name)
        if os.path.isfile(path): out.append((path, str(pid), camid))
        else: missing += 1
    print(f'VeRi train: {len(out):,} images, {len(set(r[1] for r in out))} IDs, missing={missing}')
    return out

def read_veri_split(image_dir, split_name):
    out=[]; skipped=0
    for path in sorted(glob.glob(os.path.join(image_dir, '*'))):
        if not os.path.isfile(path): continue
        m = re.match(r'^(\d+)_c(\d+)', os.path.basename(path), re.I)
        if not m: skipped += 1; continue
        out.append((path, str(m.group(1)), int(m.group(2))))
    print(f'VeRi {split_name}: {len(out):,} images, {len(set(r[1] for r in out))} IDs, skipped={skipped}')
    return out

VERI_TRAIN = read_veri_train(os.path.join(VERI_ROOT, 'train_label.xml'), os.path.join(VERI_ROOT, 'image_train'))
VERI_QUERY = read_veri_split(os.path.join(VERI_ROOT, 'image_query'), 'query')
VERI_GALLERY = read_veri_split(os.path.join(VERI_ROOT, 'image_test'), 'gallery')
assert VERI_TRAIN and VERI_QUERY and VERI_GALLERY, 'VeRi splits are empty; check dataset paths and archive structure.'

## 2. AI City Challenge 2021 Track 2 — CityFlowV2-ReID

Используем только официальный **train split**: `image_train/`, `train_label.xml`, `train_track.txt`. Файлы `image_test/`, `image_query/`, `test_label.xml`, `query_label.xml` в обучении не используются.

Укажи `AICITY_ROOT` на распакованную папку CityFlowV2-ReID. Архив изображений (около 2 ГБ) должен быть распакован отдельно; небольшой metadata ZIP, который мы проверили, содержит разметку и track-файлы, но не сами изображения.

Важная особенность: `train_track.txt` задаёт группы кадров одного vehicle ID в пределах одной камеры. Мы проверяем согласованность каждой группы с `train_label.xml` и используем track ID в sampler. Валидационные vehicle ID полностью исключаются из обучения; для каждого такого ID query берётся с одной камеры, gallery — с остальных.

In [ ]:
def read_aicity_train(root):
    image_dir = os.path.join(root, 'image_train')
    label_path = os.path.join(root, 'train_label.xml')
    track_path = os.path.join(root, 'train_track.txt')
    for p in (image_dir, label_path, track_path):
        if not os.path.exists(p):
            raise FileNotFoundError(f'Не найден компонент AI City: {p}')

    tree = ET.parse(label_path)  # ElementTree читает XML с encoding из заголовка (GB2312)
    items = tree.getroot().find('Items')
    if items is None:
        raise RuntimeError('В train_label.xml нет узла Items')

    labels = {}
    for item in items:
        name = item.attrib.get('imageName')
        pid = item.attrib.get('vehicleID')
        cam = item.attrib.get('cameraID', '')
        if not name or pid is None:
            continue
        match = re.search(r'(\d+)', cam)
        if not match:
            raise ValueError(f'Не удалось прочитать cameraID={cam!r} для {name}')
        labels[name] = (f'aicity_{pid}', int(match.group(1)))

    track_by_name = {}
    with open(track_path, 'r', encoding='utf-8') as f:
        track_lines = [line.strip() for line in f if line.strip()]

    inconsistent = []
    for track_idx, line in enumerate(track_lines):
        names = line.split()
        missing_labels = [name for name in names if name not in labels]
        if missing_labels:
            raise ValueError(f'train_track.txt: имена отсутствуют в train_label.xml: {missing_labels[:5]}')
        identity_camera = {labels[name] for name in names}
        if len(identity_camera) != 1:
            inconsistent.append((track_idx, identity_camera))
        for name in names:
            if name in track_by_name:
                raise ValueError(f'Изображение {name} встречается более чем в одном треке')
            track_by_name[name] = track_idx

    if inconsistent:
        raise ValueError(f'Найдены несогласованные треки: {inconsistent[:5]}')
    if set(track_by_name) != set(labels):
        missing_tracks = set(labels) - set(track_by_name)
        raise ValueError(f'{len(missing_tracks)} train изображений отсутствуют в train_track.txt')

    records, track_ids = [], []
    missing_images = 0
    for name, (pid, cam) in labels.items():
        path = os.path.join(image_dir, name)
        if not os.path.isfile(path):
            missing_images += 1
            continue
        records.append((path, pid, cam))
        track_ids.append(track_by_name[name])

    if not records:
        raise RuntimeError('Не найдено ни одного изображения в image_train/. Проверь AICITY_ROOT.')
    if missing_images:
        print(f'WARNING: отсутствуют изображения: {missing_images}')
    id_cameras = defaultdict(set)
    for _, pid, cam in records:
        id_cameras[pid].add(cam)

    print(f'AI City train: {len(records):,} images, {len(id_cameras)} IDs, '
          f'{len(set(r[2] for r in records))} cameras, {len(track_lines)} tracks')
    print(f'Камер на ID: min={min(map(len, id_cameras.values()))}, '
          f'median={int(np.median([len(x) for x in id_cameras.values()]))}, '
          f'max={max(map(len, id_cameras.values()))}')
    return records, track_ids

AICITY_ALL, AICITY_TRACK_IDS = read_aicity_train(AICITY_ROOT)

def split_aicity_by_identity(records, track_ids, val_fraction=0.20, seed=SEED):
    ids = sorted(set(r[1] for r in records))
    rng = random.Random(seed)
    rng.shuffle(ids)
    n_val = max(1, int(round(len(ids) * val_fraction)))
    val_ids = set(ids[:n_val])

    train_records, train_track_ids, val_records = [], [], []
    for record, track_id in zip(records, track_ids):
        if record[1] in val_ids:
            val_records.append(record)
        else:
            train_records.append(record)
            train_track_ids.append(track_id)

    # Для каждого validation ID выбираем одну query-камеру;
    # все остальные камеры этого ID образуют gallery.
    by_pid = defaultdict(list)
    for record in val_records:
        by_pid[record[1]].append(record)

    val_query, val_gallery = [], []
    for pid, pid_records in by_pid.items():
        cameras = sorted(set(r[2] for r in pid_records))
        if len(cameras) < 2:
            continue
        # Детерминированно, но не одна и та же камера для всех ID.
        query_cam = cameras[(sum(ord(ch) for ch in pid) + seed) % len(cameras)]
        val_query.extend(r for r in pid_records if r[2] == query_cam)
        val_gallery.extend(r for r in pid_records if r[2] != query_cam)

    train_ids = set(r[1] for r in train_records)
    query_ids = set(r[1] for r in val_query)
    gallery_ids = set(r[1] for r in val_gallery)
    assert not (train_ids & val_ids), 'AI City train/validation ID leakage!'
    assert query_ids == gallery_ids, 'Каждый validation ID должен иметь query и cross-camera gallery.'

    print(f'AI City split: train={len(train_records):,} images/{len(train_ids)} IDs; '
          f'validation={len(val_records):,} images/{len(val_ids)} IDs')
    print(f'Cross-camera validation: query={len(val_query):,}, gallery={len(val_gallery):,}, '
          f'positive IDs={len(query_ids)}')
    return train_records, train_track_ids, val_query, val_gallery

AICITY_TRAIN_RAW, AICITY_TRAIN_TRACK_IDS, AICITY_VAL_QUERY, AICITY_VAL_GALLERY = split_aicity_by_identity(
    AICITY_ALL, AICITY_TRACK_IDS, val_fraction=VAL_FRACTION
)

## 2. Parking-140: scan и split без утечки по ID

Ожидаемая структура — `PARKING_ROOT/ID 1/cam1_001.jpg`, `PARKING_ROOT/ID 1/cam2_001.jpg`, … `cam9_*.jpg`. Камеры, используемые для первичной регистрации, задаются ниже. **Проверь этот список под свою реальную систему.**

In [ ]:
PARKING_LAYOUT = 'folders'  # 'folders' или 'filename'
FILENAME_REGEX = re.compile(r'(?P<pid>.+?).*?cam[_-]?(?P<cam>\d+)', re.I)
PARKING_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
REGISTRATION_CAMERAS = {1, 2, 3}  # поменяй, если регистрация идёт с других камер


def infer_camera(filename):
    m = re.search(r'(?:^|[_-])cam[_-]?(\d+)(?=[_-]|\.|$)', filename, re.I)
    if not m: m = re.search(r'(?:^|[_-])c[_-]?(\d+)(?=[_-]|\.|$)', filename, re.I)
    return int(m.group(1)) if m else 0

def scan_parking(root, layout='folders'):
    if not os.path.isdir(root): raise FileNotFoundError(f'Не найден PARKING_ROOT: {root}')
    raw=[]; skipped=0
    if layout == 'folders':
        for pid_name in sorted(os.listdir(root)):
            d=os.path.join(root, pid_name)
            if not os.path.isdir(d): continue
            for p in sorted(glob.glob(os.path.join(d, '**', '*'), recursive=True)):
                if os.path.isfile(p) and os.path.splitext(p)[1].lower() in PARKING_EXTS:
                    raw.append((p, pid_name, infer_camera(os.path.basename(p))))
    elif layout == 'filename':
        for p in sorted(glob.glob(os.path.join(root, '**', '*'), recursive=True)):
            if not os.path.isfile(p) or os.path.splitext(p)[1].lower() not in PARKING_EXTS: continue
            m=FILENAME_REGEX.search(os.path.basename(p))
            if not m: skipped += 1; continue
            raw.append((p, m.group('pid').strip(), int(m.group('cam'))))
    else: raise ValueError("PARKING_LAYOUT must be 'folders' or 'filename'")
    if not raw: raise RuntimeError('Parking images not found. Check PARKING_ROOT/PARKING_LAYOUT.')
    names=sorted(set(r[1] for r in raw)); pid_map={name:i for i,name in enumerate(names)}
    records=[(p, pid_map[pid], int(cam)) for p,pid,cam in raw]
    cam_counts=Counter(r[2] for r in records)
    print(f'Parking: {len(records):,} images, {len(names)} vehicle IDs, skipped={skipped}')
    print('Camera counts:', dict(sorted(cam_counts.items())))
    if cam_counts.get(0, 0):
        print(f'WARNING: {cam_counts[0]} files have unknown camera ID. Fix filename patterns before trusting cross-camera metrics.')
    print('Example folder/ID mapping:', list(pid_map.items())[:8])
    return records, pid_map

PARKING_ALL, PARKING_PID_MAP = scan_parking(PARKING_ROOT, PARKING_LAYOUT)

def split_by_identity(records, val_fraction=0.20, seed=SEED):
    ids=sorted(set(r[1] for r in records)); rng=random.Random(seed); rng.shuffle(ids)
    n_val=max(1, int(round(len(ids)*val_fraction)))
    val_ids=set(ids[:n_val]); train=[r for r in records if r[1] not in val_ids]; val=[r for r in records if r[1] in val_ids]
    print(f'Parking split by ID: train IDs={len(set(r[1] for r in train))}, val IDs={len(set(r[1] for r in val))}; overlap={len(set(r[1] for r in train)&set(r[1] for r in val))}')
    assert not (set(r[1] for r in train)&set(r[1] for r in val)), 'ID leakage!'
    return train, val

PARKING_TRAIN_RAW, PARKING_VAL_RAW = split_by_identity(PARKING_ALL, val_fraction=0.20)

## 3. Training settings

Параметры умеренные для небольшого Parking-140. Если не хватает VRAM, снизь `BATCH_P`/`BATCH_K` или размер картинки. В batch желательно иметь несколько разных ID, а для каждого ID — кадры разных камер.

In [ ]:
HEIGHT, WIDTH = 384, 192
BATCH_P = 16                 # число ID в batch
BATCH_K = 4                  # изображений на ID; итого batch=64
BATCH_SIZE = BATCH_P * BATCH_K
WORKERS = 2
VERI_EPOCHS = 12
AICITY_EPOCHS = 8
RUN_AICITY_STAGE = True      # False — пропустить этап, если checkpoint уже обучен
PARKING_INIT_SOURCE = 'aicity'  # 'aicity' или 'veri' для сравнения двух вариантов
PARKING_EPOCHS = 20
LR_VERI = 3e-4
LR_AICITY = 1e-4
LR_PARKING = 3e-5
WEIGHT_DECAY = 5e-4
TRIPLET_MARGIN = 0.30
VAL_FRACTION = 0.20
train_tfms = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.18, 0.18, 0.18, 0.04)], p=0.5),
    transforms.Pad(10),
    transforms.RandomCrop((HEIGHT, WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02,0.15), ratio=(0.3,3.3), value='random'),
])
eval_tfms = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
class VehicleDataset(Dataset):
    def __init__(self, records, transform=None, pid_map=None):
        self.records=list(records); self.transform=transform
        self.pid_map=pid_map if pid_map is not None else {p:i for i,p in enumerate(sorted(set(r[1] for r in records)))}
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        path,pid,cam=self.records[idx]
        try:
            img=Image.open(path).convert('RGB')
        except Exception as e:
            raise RuntimeError(f'Cannot read image: {path}: {e}')
        if self.transform: img=self.transform(img)
        return img, self.pid_map[pid], int(cam), path
class CameraAwarePKSampler(Sampler):
    """P IDs x K images: distinct cameras first, then distinct tracks, then replacement."""
    def __init__(self, records, p=16, k=4, seed=SEED, track_ids=None):
        self.records = list(records)
        self.p, self.k, self.seed, self.epoch = p, k, seed, 0
        if track_ids is None:
            track_ids = list(range(len(records)))
        if len(track_ids) != len(records):
            raise ValueError('track_ids должен иметь ту же длину, что и records')
        self.by_pid_cam_track = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
        for idx, (record, track_id) in enumerate(zip(self.records, track_ids)):
            pid, cam = record[1], record[2]
            self.by_pid_cam_track[pid][cam][track_id].append(idx)
        self.pids = list(self.by_pid_cam_track)
        self.num_batches = max(1, math.ceil(len(self.records) / (self.p * self.k)))
    def __len__(self):
        return self.num_batches
    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        self.epoch += 1
        for _ in range(self.num_batches):
            chosen = rng.sample(self.pids, self.p) if len(self.pids) >= self.p else [rng.choice(self.pids) for _ in range(self.p)]
            batch = []
            for pid in chosen:
                by_cam = self.by_pid_cam_track[pid]
                cameras = list(by_cam)
                rng.shuffle(cameras)
                picked, used_tracks, candidates = [], set(), []
                # Сначала хотя бы один отдельный трек с каждой доступной камеры.
                for cam in cameras:
                    tracks = list(by_cam[cam].items())
                    rng.shuffle(tracks)
                    track_id, indices = tracks[0]
                    picked.append(rng.choice(indices))
                    used_tracks.add((cam, track_id))
                    candidates.extend((cam, tid, idxs) for tid, idxs in tracks[1:])
                    if len(picked) == self.k:
                        break
                # Затем добираем другие треки (включая дополнительные треки той же камеры).
                rng.shuffle(candidates)
                for cam, track_id, indices in candidates:
                    if len(picked) >= self.k:
                        break
                    if (cam, track_id) in used_tracks:
                        continue
                    picked.append(rng.choice(indices))
                    used_tracks.add((cam, track_id))
                # Только если уникальных камер/треков недостаточно — разрешаем повтор.
                all_indices = [idx for cam in by_cam for tid in by_cam[cam] for idx in by_cam[cam][tid]]
                while len(picked) < self.k:
                    picked.append(rng.choice(all_indices))
                batch.extend(picked)
            yield batch
def make_train_loader(records, pid_map, track_ids=None):
    ds = VehicleDataset(records, train_tfms, pid_map)
    sampler = CameraAwarePKSampler(records, BATCH_P, BATCH_K, track_ids=track_ids)
    return DataLoader(ds, batch_sampler=sampler, num_workers=WORKERS, pin_memory=torch.cuda.is_available())
def make_eval_loader(records, pid_map=None):
    ds=VehicleDataset(records, eval_tfms, pid_map=pid_map)
    return DataLoader(ds, batch_size=128, shuffle=False, num_workers=WORKERS, pin_memory=torch.cuda.is_available())

## 4. Модель, loss и retrieval-метрики

Используем `osnet_ain_x1_0` с ImageNet-pretrained весами. В VeRi учим классификацию ID + batch-hard triplet; для Parking переносим только совместимые веса backbone, так как классы автомобилей другие.

In [ ]:
def build_model(num_classes, pretrained=False):
    model=torchreid.models.build_model(name='osnet_ain_x1_0', num_classes=num_classes, loss='triplet', pretrained=pretrained)
    return model.to(DEVICE)

def unpack_train_output(out):
    # torchreid OSNet with loss='triplet' usually returns (logits, embeddings) in train mode.
    if isinstance(out, (tuple,list)) and len(out)>=2: return out[0], out[1]
    raise RuntimeError('Model did not return (logits, embeddings). Check torchreid version/model API.')

def batch_hard_triplet(embeddings, labels, margin=TRIPLET_MARGIN):
    z=F.normalize(embeddings, p=2, dim=1)
    dist=(1.0 - z @ z.t()).clamp(min=0)
    same=labels[:,None].eq(labels[None,:])
    eye=torch.eye(len(labels), device=labels.device, dtype=torch.bool)
    pos_mask=same & ~eye; neg_mask=~same
    valid=pos_mask.any(dim=1) & neg_mask.any(dim=1)
    if not valid.any(): return embeddings.new_tensor(0.0)
    hardest_pos=dist.masked_fill(~pos_mask, -1e4).max(dim=1).values
    hardest_neg=dist.masked_fill(~neg_mask, 1e4).min(dim=1).values
    return F.relu(hardest_pos[valid]-hardest_neg[valid]+margin).mean()

def train_one_epoch(model, loader, optimizer, scaler=None):
    model.train(); total=0.0; ce_total=0.0; tri_total=0.0; n=0
    for images, labels, cams, paths in loader:
        images=images.to(DEVICE, non_blocking=True); labels=labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16 if DEVICE.type=='cuda' else torch.bfloat16, enabled=(DEVICE.type=='cuda')):
            logits, embeddings=unpack_train_output(model(images))
            ce=F.cross_entropy(logits, labels, label_smoothing=0.05)
            tri=batch_hard_triplet(embeddings, labels)
            loss=ce+tri
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        b=len(labels); total+=loss.item()*b; ce_total+=ce.item()*b; tri_total+=tri.item()*b; n+=b
    return {'loss':total/max(n,1), 'ce':ce_total/max(n,1), 'triplet':tri_total/max(n,1)}

@torch.no_grad()
def extract_features(model, records, pid_map=None):
    if pid_map is None:
        all_pids=sorted(set(r[1] for r in records), key=str)
        pid_map={pid:i for i,pid in enumerate(all_pids)}
    loader=make_eval_loader(records, pid_map=pid_map); model.eval(); feats=[]; pids=[]; cams=[]; paths=[]
    for images, labels, camids, batch_paths in loader:
        images=images.to(DEVICE, non_blocking=True)
        out=model(images)
        if isinstance(out,(tuple,list)): out=out[-1]
        out=F.normalize(out, p=2, dim=1)
        feats.append(out.cpu().numpy()); pids.extend(labels.numpy().tolist()); cams.extend(camids.numpy().tolist()); paths.extend(list(batch_paths))
    return np.concatenate(feats), np.asarray(pids), np.asarray(cams), paths

def retrieval_metrics(model, query_records, gallery_records):
    """VeRi-style retrieval metrics: ignore same-PID/same-camera gallery items."""
    if not query_records or not gallery_records:
        return {'rank1':float('nan'),'rank5':float('nan'),'mAP':float('nan'),'queries':0}
    all_records=list(query_records)+list(gallery_records)
    shared_pid_map={pid:i for i,pid in enumerate(sorted(set(r[1] for r in all_records), key=str))}
    qf,qp,qc,_=extract_features(model, query_records, shared_pid_map)
    gf,gp,gc,_=extract_features(model, gallery_records, shared_pid_map)
    sims=qf @ gf.T
    aps=[]; r1=[]; r5=[]; skipped_no_positive=0
    for i,pid in enumerate(qp):
        # Standard Re-ID protocol: remove gallery images of the same identity
        # captured by the same camera as this query.
        valid = ~((gp == pid) & (gc == qc[i]))
        good = (gp == pid) & valid
        if not good.any():
            skipped_no_positive += 1
            continue
        scores = sims[i].copy()
        scores[~valid] = -np.inf
        order=np.argsort(-scores, kind='stable')
        matches=good[order]
        first=np.flatnonzero(matches)[0]
        r1.append(float(first == 0)); r5.append(float(first < 5))
        hit=np.flatnonzero(matches)
        precision=(np.arange(1,len(hit)+1)/(hit+1)).mean()
        aps.append(float(precision))
    return {
        'rank1':float(np.mean(r1)) if r1 else float('nan'),
        'rank5':float(np.mean(r5)) if r5 else float('nan'),
        'mAP':float(np.mean(aps)) if aps else float('nan'),
        'queries':len(aps),
        'queries_skipped_no_cross_camera_positive':skipped_no_positive,
    }

def load_backbone_weights(model, checkpoint_path):
    obj=torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state=obj.get('state_dict', obj.get('model', obj.get('model_state_dict', obj))) if isinstance(obj,dict) else obj
    own=model.state_dict(); good={};
    for key,value in state.items():
        key=key.removeprefix('module.')
        if key in own and own[key].shape==value.shape and not key.startswith('classifier'):
            good[key]=value
    missing,unexpected=model.load_state_dict(good, strict=False)
    print(f'Loaded compatible non-classifier tensors: {len(good)}; checkpoint keys={len(state)}')
    return model

## 5. Этап A — полноценное обучение на VeRi-776

Для выбора checkpoint используем официальный query/gallery VeRi. Если GPU слабый, сначала поставь `VERI_EPOCHS=5`, но не пропускай этот этап без причины.

In [ ]:
veri_pid_map={pid:i for i,pid in enumerate(sorted(set(r[1] for r in VERI_TRAIN)))}
veri_loader=make_train_loader(VERI_TRAIN, veri_pid_map)
model=build_model(num_classes=len(veri_pid_map), pretrained=True)
optimizer=torch.optim.AdamW(model.parameters(), lr=LR_VERI, weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=VERI_EPOCHS, eta_min=LR_VERI*0.05)

veri_best=-1.0
VERI_CKPT=os.path.join(SAVE_ROOT,'osnet_ain_veri_best.pth')
veri_history=[]
for epoch in range(1, VERI_EPOCHS+1):
    stats=train_one_epoch(model, veri_loader, optimizer)
    scheduler.step()
    metrics=retrieval_metrics(model, VERI_QUERY, VERI_GALLERY)
    row={'epoch':epoch, **stats, **metrics, 'lr':optimizer.param_groups[0]['lr']}
    veri_history.append(row)
    print(f"[VeRi {epoch:02d}/{VERI_EPOCHS}] loss={stats['loss']:.4f} CE={stats['ce']:.4f} tri={stats['triplet']:.4f} mAP={metrics['mAP']:.4f} R1={metrics['rank1']:.4f} R5={metrics['rank5']:.4f}")
    score=metrics['mAP'] if np.isfinite(metrics['mAP']) else metrics['rank1']
    if score>veri_best:
        veri_best=score
        torch.save({'state_dict':model.state_dict(),'epoch':epoch,'metrics':metrics,'num_classes':len(veri_pid_map)}, VERI_CKPT)
        print('  saved best:', VERI_CKPT)
print('Best VeRi mAP:', veri_best)

## 6. Этап B — fine-tuning на AI City Track 2

Здесь обучаемся только на train ID. Validation ID полностью отделены от train и используются только для выбора checkpoint. Query и gallery каждого validation автомобиля взяты из разных камер.

Sampler для AI City учитывает и camera ID, и track ID: не заполняет batch большим количеством почти одинаковых соседних кадров из одного трека.

In [ ]:
aicity_history = []
AICITY_CKPT = os.path.join(SAVE_ROOT, 'osnet_ain_veri_aicity_best.pth')

if RUN_AICITY_STAGE:
    aicity_train_pids = sorted(set(r[1] for r in AICITY_TRAIN_RAW))
    aicity_pid_map = {pid:i for i,pid in enumerate(aicity_train_pids)}
    aicity_loader = make_train_loader(AICITY_TRAIN_RAW, aicity_pid_map, AICITY_TRAIN_TRACK_IDS)

    model_aicity = build_model(num_classes=len(aicity_pid_map), pretrained=False)
    load_backbone_weights(model_aicity, VERI_CKPT)

    # Зафиксируем перенос VeRi -> AI City до fine-tuning.
    aicity_before = retrieval_metrics(model_aicity, AICITY_VAL_QUERY, AICITY_VAL_GALLERY)
    print('AI City validation before fine-tuning:', aicity_before)

    optimizer = torch.optim.AdamW(model_aicity.parameters(), lr=LR_AICITY, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=AICITY_EPOCHS, eta_min=LR_AICITY * 0.05
    )

    aicity_best = -1.0
    for epoch in range(1, AICITY_EPOCHS + 1):
        stats = train_one_epoch(model_aicity, aicity_loader, optimizer)
        scheduler.step()
        metrics = retrieval_metrics(model_aicity, AICITY_VAL_QUERY, AICITY_VAL_GALLERY)
        row = {'epoch':epoch, **stats, **metrics, 'lr':optimizer.param_groups[0]['lr']}
        aicity_history.append(row)
        print(f"[AI City {epoch:02d}/{AICITY_EPOCHS}] loss={stats['loss']:.4f} "
              f"CE={stats['ce']:.4f} tri={stats['triplet']:.4f} "
              f"mAP={metrics['mAP']:.4f} R1={metrics['rank1']:.4f} "
              f"R5={metrics['rank5']:.4f} queries={metrics['queries']}")
        score = metrics['mAP'] if np.isfinite(metrics['mAP']) else metrics['rank1']
        if score > aicity_best:
            aicity_best = score
            torch.save({
                'state_dict':model_aicity.state_dict(),
                'epoch':epoch,
                'metrics':metrics,
                'num_classes':len(aicity_pid_map),
                'train_ids':len(aicity_train_pids),
                'validation_protocol':'identity-disjoint, one query camera per ID, gallery from other cameras'
            }, AICITY_CKPT)
            print('  saved best:', AICITY_CKPT)

    print('Best AI City cross-camera validation mAP:', aicity_best)
else:
    print('AI City training skipped; using an existing checkpoint only if PARKING_INIT_SOURCE="aicity".')

## 7. Этап C — fine-tuning на Parking-140

Train/validation ID не пересекаются. Parking validation строится по твоему реальному протоколу: gallery — регистрационные камеры `REGISTRATION_CAMERAS`, query — остальные камеры.

Инициализация берётся из лучшего AI City checkpoint. Если эксперимент AI City окажется хуже базовой VeRi модели, сравни обе версии на одном и том же Parking split; не выбирай checkpoint только по AI City mAP.

In [ ]:
# Для Parking train классы перенумеровываются с нуля.

parking_train_pids=sorted(set(r[1] for r in PARKING_TRAIN_RAW))

parking_pid_map={pid:i for i,pid in enumerate(parking_train_pids)}

parking_loader=make_train_loader(PARKING_TRAIN_RAW, parking_pid_map)



model_parking=build_model(num_classes=len(parking_pid_map), pretrained=False)

if PARKING_INIT_SOURCE == 'aicity':
    PARKING_INIT_CKPT = AICITY_CKPT
elif PARKING_INIT_SOURCE == 'veri':
    PARKING_INIT_CKPT = VERI_CKPT
else:
    raise ValueError("PARKING_INIT_SOURCE должен быть 'aicity' или 'veri'")
if not os.path.isfile(PARKING_INIT_CKPT):
    raise FileNotFoundError(f'Не найден checkpoint для Parking: {PARKING_INIT_CKPT}')
load_backbone_weights(model_parking, PARKING_INIT_CKPT)

# Малый LR для backbone: не разрушать признаки, выученные на VeRi.

optimizer=torch.optim.AdamW(model_parking.parameters(), lr=LR_PARKING, weight_decay=WEIGHT_DECAY)

scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PARKING_EPOCHS, eta_min=LR_PARKING*0.05)



val_gallery=[r for r in PARKING_VAL_RAW if r[2] in REGISTRATION_CAMERAS]

val_query=[r for r in PARKING_VAL_RAW if r[2] not in REGISTRATION_CAMERAS and r[2] != 0]

val_gallery_ids=set(r[1] for r in val_gallery); val_query_ids=set(r[1] for r in val_query)

known_ids=val_gallery_ids & val_query_ids

unknown_query_ids=val_query_ids - val_gallery_ids

print('Registration cameras:', sorted(REGISTRATION_CAMERAS))

print(f'Parking validation: gallery={len(val_gallery)} imgs/{len(val_gallery_ids)} IDs; query={len(val_query)} imgs/{len(val_query_ids)} IDs')

print(f'IDs with positive gallery+query={len(known_ids)}; query-only IDs (unknown to gallery)={len(unknown_query_ids)}')

if not val_gallery or not val_query: raise RuntimeError('Parking validation gallery/query empty. Check camera IDs and filename camera parsing.')

if len(known_ids)<2: print('WARNING: too few cross-camera validation IDs; metrics will be unstable. Check REGISTRATION_CAMERAS.')



PARKING_CKPT=os.path.join(SAVE_ROOT,f'osnet_ain_parking_from_{PARKING_INIT_SOURCE}_best.pth')

parking_best=-1.0

parking_history=[]

for epoch in range(1, PARKING_EPOCHS+1):

    stats=train_one_epoch(model_parking, parking_loader, optimizer)

    scheduler.step()

    metrics=retrieval_metrics(model_parking, val_query, val_gallery)

    row={'epoch':epoch, **stats, **metrics, 'lr':optimizer.param_groups[0]['lr']}

    parking_history.append(row)

    print(f"[Parking {epoch:02d}/{PARKING_EPOCHS}] loss={stats['loss']:.4f} CE={stats['ce']:.4f} tri={stats['triplet']:.4f} mAP={metrics['mAP']:.4f} R1={metrics['rank1']:.4f} R5={metrics['rank5']:.4f} queries={metrics['queries']}")

    score=metrics['mAP'] if np.isfinite(metrics['mAP']) else metrics['rank1']

    if score>parking_best:

        parking_best=score

        torch.save({'state_dict':model_parking.state_dict(),'epoch':epoch,'metrics':metrics,'num_classes':len(parking_pid_map),'registration_cameras':sorted(REGISTRATION_CAMERAS),'init_source':PARKING_INIT_SOURCE}, PARKING_CKPT)

        print('  saved best:', PARKING_CKPT)

print('Best Parking validation mAP:', parking_best)

## 8. Open-set проверка: когда правильного ID нет в gallery

Одного mAP мало: в tracking опасно передать автомобиль неправильному ID. Считаем genuine и impostor/unknown similarity и диагностический порог на Parking validation. Порог не является production threshold — проверь его на отдельном видео с размеченными переходами камер.

In [ ]:
@torch.no_grad()
def open_set_report(model, query_records, gallery_records):
    all_records=list(query_records)+list(gallery_records)
    shared_pid_map={pid:i for i,pid in enumerate(sorted(set(r[1] for r in all_records), key=str))}
    qf,qp,qc,qpaths=extract_features(model, query_records, shared_pid_map)
    gf,gp,gc,gpaths=extract_features(model, gallery_records, shared_pid_map)
    sims=qf @ gf.T
    best_idx=sims.argmax(axis=1); best_score=sims.max(axis=1); best_pid=gp[best_idx]
    known=np.isin(qp, np.unique(gp))
    correct=(best_pid==qp) & known
    genuine=[]; impostor=[]; details=[]
    for i,pid in enumerate(qp):
        pos=sims[i][gp==pid]
        neg=sims[i][gp!=pid]
        if len(pos):
            genuine.append(float(pos.max()))
            if len(neg): impostor.append(float(neg.max()))
        else:
            impostor.append(float(sims[i].max()))
        details.append({'query':qpaths[i], 'true_pid':int(pid), 'pred_pid':int(best_pid[i]), 'best_similarity':float(best_score[i]), 'has_gallery_match':bool(known[i]), 'top1_correct':bool(correct[i])})
    # Threshold sweep: genuine scores should be accepted; strongest wrong/unknown score should be rejected.
    candidates=np.linspace(0.20,0.95,151)
    best_thr=0.65; best_bal=-1
    if genuine and impostor:
        for thr in candidates:
            tpr=np.mean(np.asarray(genuine)>=thr)
            tnr=np.mean(np.asarray(impostor)<thr)
            bal=(tpr+tnr)/2
            if bal>best_bal: best_bal=bal; best_thr=float(thr)
    accepted=best_score>=best_thr
    known_n=int(known.sum()); unknown_n=int((~known).sum())
    report={
      'queries_total':int(len(qp)), 'known_queries':known_n, 'unknown_queries':unknown_n,
      'known_top1_accuracy':float(np.mean(correct[known])) if known.any() else float('nan'),
      'unknown_rejection_rate':float(np.mean(~accepted[~known])) if (~known).any() else float('nan'),
      'false_accept_rate_on_unknown':float(np.mean(accepted[~known])) if (~known).any() else float('nan'),
      'calibrated_threshold_diagnostic_only':best_thr,
      'genuine_best_similarity_mean':float(np.mean(genuine)) if genuine else float('nan'),
      'hard_impostor_similarity_mean':float(np.mean(impostor)) if impostor else float('nan'),
    }
    return report, details

# Load the best Parking checkpoint, not the last epoch.
obj=torch.load(PARKING_CKPT, map_location='cpu', weights_only=False)
model_parking.load_state_dict(obj['state_dict'])
open_report, open_details=open_set_report(model_parking, val_query, val_gallery)
print(json.dumps(open_report, indent=2, ensure_ascii=False))
with open(os.path.join(SAVE_ROOT,'parking_open_set_report.json'),'w',encoding='utf-8') as f:
    json.dump({'report':open_report,'details':open_details},f,indent=2,ensure_ascii=False)

## 9. Сохранить результаты и использовать checkpoint

В `SAVE_ROOT` сохраняются лучшие checkpoints VeRi, AI City и Parking, а также CSV-истории и open-set отчёт. Для первого теста в tracking предназначен `osnet_ain_parking_best.pth`; порог similarity нужно отдельно откалибровать на реальных переходах.

In [ ]:
import pandas as pd

pd.DataFrame(aicity_history).to_csv(os.path.join(SAVE_ROOT,'aicity_history.csv'),index=False)
pd.DataFrame(veri_history).to_csv(os.path.join(SAVE_ROOT,'veri_history.csv'),index=False)

pd.DataFrame(parking_history).to_csv(os.path.join(SAVE_ROOT,'parking_history.csv'),index=False)

print('Saved artifacts:')

for name in sorted(os.listdir(SAVE_ROOT)):

    print(os.path.join(SAVE_ROOT,name))

## 10. Ограничения и корректная интерпретация

- В обучении AI City используется только Track 2 train split. Не подключай Track 3: FAQ соревнования прямо запрещает применять Track 3 к Track 2.
- Test/query AI City не используются для обучения или выбора checkpoint.
- В официальном тесте AI City есть 6 камер, отсутствующих в train. Наша внутренняя validation проверяет перенос между камерами, уже представленными в train, но не даёт размеченной оценки на этих 6 новых камерах.
- `RandomHorizontalFlip(p=0.5)` уже присутствует в аугментациях исходного pipeline и применяется ко всем train-изображениям, не только к Parking cameras 1–3. Сравни p=0.3 и p=0.5 отдельными запусками, если хочешь проверить влияние на целевые камеры.
- Важно по лицензии: после завершения AI City Challenge данные и модели, обученные на них, разрешено использовать только в некоммерческих академических целях. Не используй checkpoint, обученный на CityFlowV2, в коммерческой/производственной системе без отдельного разрешения правообладателя.